# Etapa 1: Ingestión y Construcción del Dataset Único Mensual SIPSA

**Proyecto**: Plataforma Analítica del Mercado de la Papa en Colombia (SIPSA 2019–2025)  
**Marco Metodológico**: CRISP-DM (Etapa 1: Ingestion) & Marco PDCO (Fase DEVELOPMENT)  
**Estándares**: Clean Architecture, DAMA-BOK, PEP 8, ISO/IEC 25010  

---

### Objetivos de la Etapa:
1. **Ingestión Políglota**: Procesar los 16 periodos crudos en `data/RAW/sipsa/` (`.csv`, `.dta`, `.sav`, `.sas`).
2. **Resolución de Schema Drift**: Unificar las 6 variantes históricas de cabeceras de DANE a un esquema canónico común.
3. **Dimensionalidad Temporal Mensual**: Estandarizar transacciones intradiarias, cuatrimestres y semestres a una escala **MENSUAL** continua (`fecha_mes = YYYY-MM-01`).
4. **Regla de Invarianza Estricta**: **Cero Precios Nulos** (`precio_prom_kg IS NOT NULL` y `> 0`).
5. **Persistencia Medallion**: Exportar el dataset consolidado a `data/CLEANED/dataset_sipsa_mensual_nacional.parquet`.

In [ ]:
# ==============================================================================
# CELDA 1 OBLIGATORIA: INSTALACIÓN IDEMPOTENTE DE DEPENDENCIAS EN EL KERNEL
# ==============================================================================
import sys

%pip install --quiet --upgrade pip
%pip install --quiet \
    numpy>=1.24.0 \
    pandas>=2.0.0 \
    scipy>=1.10.0 \
    statsmodels>=0.14.0 \
    scikit-learn>=1.3.0 \
    matplotlib>=3.7.0 \
    seaborn>=0.12.0 \
    openpyxl>=3.1.0 \
    pyreadstat>=1.2.0 \
    fastparquet>=2023.8.0 \
    pyarrow>=12.0.0 \
    tabulate>=0.9.0

print(f"[OK] Dependencias verificadas con éxito en Python: {sys.version.split()[0]}")

In [ ]:
# ==============================================================================
# CONFIGURACIÓN DEL ENTORNO Y CARGA DE MÓDULOS SRC (CLEAN ARCHITECTURE)
# ==============================================================================
from pathlib import Path
import sys

# Incorporar la raíz del proyecto al PYTHONPATH
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np

from src.infrastructure.config import RAW_SIPSA_DIR, CLEANED_DIR
from src.application.ingestion_service import IngestionService
from src.domain.invariants import validar_precio_escalar

print(f"Directorio Raw SIPSA: {RAW_SIPSA_DIR}")
print(f"Directorio de Salida Cleaned: {CLEANED_DIR}")

In [ ]:
# ==============================================================================
# EJECUCIÓN DEL SERVICIO DE INGESTIÓN POLÍGLOTA Y UNIFICACIÓN MENSUAL
# ==============================================================================
service = IngestionService(raw_dir=RAW_SIPSA_DIR, output_dir=CLEANED_DIR)
df_mensual, auditoria = service.construir_dataset_unico_nacional()

print("\n--- RESUMEN DE AUDITORÍA DE INGESTIÓN ---")
print(f"Archivos procesados: {auditoria['archivos_procesados']}")
print(f"Total filas mensuales generadas: {auditoria['total_filas_mensuales']:,}")

In [ ]:
# ==============================================================================
# AUDITORÍA DE CALIDAD DAMA-BOK: VERIFICACIÓN DE CERO PRECIOS NULOS
# ==============================================================================
total_filas = len(df_mensual)
nulos_precio = df_mensual["precio_prom_kg"].isna().sum()
menores_cero = (df_mensual["precio_prom_kg"] <= 0).sum()

print("=== CONTROL DE CALIDAD DE PRECIOS ===")
print(f"Total registros en dataset: {total_filas:,}")
print(f"Precios nulos (NaN / None): {nulos_precio} (Tasa: {nulos_precio/total_filas:.4%})")
print(f"Precios <= 0: {menores_cero}")

assert nulos_precio == 0, "ERROR CRÍTICO: Se encontraron precios nulos en el dataset consolidado."
assert menores_cero == 0, "ERROR CRÍTICO: Se encontraron precios menores o iguales a cero."
print("\n[PASÓ] ¡INVARIANTE DE CERO PRECIOS NULOS CUMPLIDA AL 100%!")

In [ ]:
# ==============================================================================
# PERFILAMIENTO DEL EJE TEMPORAL MENSUAL CONTINUO
# ==============================================================================
print("=== COBERTURA TEMPORAL DEL DATASET ÚNICO ===")
resumen_anual = df_mensual.groupby("año").agg(
    meses_observados=("mes", "nunique"),
    registros_mensuales=("precio_prom_kg", "count"),
    volumen_total_ton=("volumen_ingreso_ton", "sum"),
    precio_promedio_general=("precio_prom_kg", "mean")
).reset_index()

display(resumen_anual)

print("\nPrimeros 5 registros del Dataset Único:")
display(df_mensual.head(5))

### Conclusiones de la Etapa 1:
- El Dataset Único Nacional SIPSA se ha generado y persistido en `data/CLEANED/dataset_sipsa_mensual_nacional.parquet`.
- La temporalidad mensual continua permite el encadenamiento directo con la **Etapa 2 (EDA)** para la caracterización de distribuciones y asimetría.